# Create Train, Validation, and Test Datasets

In [ ]:
# Install split-folders for organizing data into training and validation sets
!pip install split-folders

In [ ]:
# Import necessary libraries
import os
import random
import shutil
import splitfolders

In [ ]:
# Steps to create train, validation, and test datasets:
# 1. Move the image dataset to 'Bleached' and 'Unbleached' folders to a preferred local path
# 2. Specify their location in raw_image_data_path
# 3. Define a local destination for the processed image datasets
# 4. Delete previously created train_validation_test folder if it exists

# Define local paths
raw_image_data_path = '/Users/et/code/Lucia-Cordero/ReefSight-Project/raw_data/image_data/raw_image_data'
bleached_folder_path = os.path.join(raw_image_data_path, 'Bleached')
unbleached_folder_path = os.path.join(raw_image_data_path, 'Unbleached')

# Define destination folder for processed datasets
train_validation_test_target_folder = '/Users/et/code/Lucia-Cordero/ReefSight-Project/raw_data/image_data/'
train_validation_test_folder = os.path.join(train_validation_test_target_folder, 'train_validation_test')

# Flag to randomly delete images for balancing the dataset
rand_del_images = True

In [ ]:
# Randomly delete images from image folder that contains more images, ensuring balanced image dataset
if rand_del_images:
    # Get the list of images in each folder
    bleached_images = os.listdir(bleached_folder_path)
    unbleached_images = os.listdir(unbleached_folder_path)

    # Determine the number of images in each folder
    num_bleached = len(bleached_images)
    num_unbleached = len(unbleached_images)

    # Find the target number of images
    target_count = min(num_bleached, num_unbleached)

    # If the bleached folder has more images, delete random ones
    if num_bleached > num_unbleached:
        images_to_remove = random.sample(bleached_images, num_bleached - target_count)
        for image in images_to_remove:
            os.remove(os.path.join(bleached_folder_path, image))
        print(f"Deleted {len(images_to_remove)} images from the Bleached folder.")

    # If the unbleached folder has more images, delete random ones
    elif num_unbleached > num_bleached:
        images_to_remove = random.sample(unbleached_images, num_unbleached - target_count)
        for image in images_to_remove:
            os.remove(os.path.join(unbleached_folder_path, image))
        print(f"Deleted {len(images_to_remove)} images from the Unbleached folder.")
    else:
        print("Both folders already have the same number of images.")

# Function to create train, validation, and test folder with images
def create_train_validation_test_split(raw_image_data_path, train_validation_test_folder, ratio=(0.6, 0.2, 0.2)):
    """
    Splits a dataset of images into training, validation, and test sets.

    Parameters:
        raw_image_data_path (str): Path to the folder containing raw images.
        train_validation_test_folder (str): Directory to store the split datasets.
        ratio (tuple): Proportions for train, validation, and test sets.

    The function checks for existing output folders, combines images, splits them according
    to the specified ratio, and cleans up temporary directories.
    """

    # Check if the train_validation_test_folder already exists
    if os.path.exists(train_validation_test_folder):
        print(f"The folder '{train_validation_test_folder}' already exists. Please remove it before running the split.")
        return None

    # Create a data directory for combined images
    combined_folder = os.path.join(raw_image_data_path, "combined")
    os.makedirs(combined_folder, exist_ok=True)

    # Move the images into the combined folder
    for category in ['Bleached', 'Unbleached']:
        source_folder = os.path.join(raw_image_data_path, category)
        target_folder = os.path.join(combined_folder, category)
        shutil.copytree(source_folder, target_folder)

    # Split the data into train, validation, and test folders with appropriate ratios
    splitfolders.ratio(combined_folder, output=train_validation_test_folder, seed=42, ratio=ratio)

    # Clean up the combined folder (optional)
    shutil.rmtree(combined_folder)

    print("Data split and organization completed!")

    # Return the paths to each split
    return {
        'train_set': os.path.join(train_validation_test_folder, "train"),
        'validation_set': os.path.join(train_validation_test_folder, "val"),
        'test_set': os.path.join(train_validation_test_folder, "test")
    }

# Call function to create the splits
dataset_paths = create_train_validation_test_split(raw_image_data_path, train_validation_test_folder, ratio=(0.6, 0.2, 0.2))

print("\nCopy this to model notebooks:")
print(f"train_data_dir = '{dataset_paths['train_set']}'")
print(f"val_data_dir = '{dataset_paths['validation_set']}'")
print(f"test_data_dir = '{dataset_paths['test_set']}'\n")